# Designing Data Structures: LRU Cache, Autocomplete and More

Notebook for **Chapter 11** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 How to Talk Through a Design Question

## 11.2 LRU Cache, OrderedDict and LFU Awareness

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.data = OrderedDict()           # oldest first, newest last

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)          # reading counts as using
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.cap:
            self.data.popitem(last=False)   # drop the oldest

c = LRUCache(2)
c.put(1, 10)
c.put(2, 20)
print(c.get(1))
c.put(3, 30)
print(c.get(2))
print(list(c.data.items()))

10
-1
[(1, 10), (3, 30)]


In [ ]:
class Node:
    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}                          # key -> Node
        self.head, self.tail = Node(), Node()  # sentinels
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):                   # unlink in O(1)
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_front(self, node):                # newest sits right after head
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_front(node)
        return node.val

    def put(self, key, value):
        if self.cap == 0:
            return
        if key in self.map:                    # update value and refresh
            node = self.map[key]
            node.val = value
            self._remove(node)
            self._add_front(node)
            return
        if len(self.map) == self.cap:          # full: evict the oldest
            oldest = self.tail.prev
            self._remove(oldest)
            del self.map[oldest.key]
        node = Node(key, value)
        self.map[key] = node
        self._add_front(node)

c = LRUCache(2)
out = []
c.put(1, 10); c.put(2, 20)
out.append(c.get(1))
c.put(3, 30)
out.append(c.get(2))
c.put(4, 40)
out += [c.get(1), c.get(3), c.get(4)]
print(out)
assert out == [10, -1, -1, 30, 40]
one = LRUCache(1)
one.put(5, 1); one.put(5, 2); one.put(6, 3)
print(one.get(5), one.get(6))

[10, -1, -1, 30, 40]
-1 3


## 11.3 Stack and Queue Designs: Min Stack, Queue from Stacks, Iterators

In [ ]:
class MinStack:
    def __init__(self):
        self.items = []
        self.mins = []          # mins[i] = smallest of items[0..i]

    def push(self, x):
        self.items.append(x)
        self.mins.append(x if not self.mins else min(x, self.mins[-1]))

    def pop(self):
        self.mins.pop()
        return self.items.pop()

    def get_min(self):
        return self.mins[-1]

s = MinStack()
for x in (5, 3, 7, 3, 8):
    s.push(x)
print(s.get_min(), s.pop(), s.pop(), s.get_min())
print(s.pop(), s.pop(), s.get_min())
assert s.items == [5]

3 8 3 3
7 3 5


In [ ]:
class MyQueue:
    def __init__(self):
        self.inbox = []     # new items land here
        self.outbox = []    # items leave from here (oldest on top)

    def push(self, x):
        self.inbox.append(x)

    def _shift(self):
        if not self.outbox:               # pour only when outbox is empty
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def pop(self):
        self._shift()
        return self.outbox.pop()

    def peek(self):
        self._shift()
        return self.outbox[-1]

    def empty(self):
        return not self.inbox and not self.outbox

q = MyQueue()
q.push(1); q.push(2); q.push(3)
print(q.pop())
q.push(4)
print(q.pop(), q.pop(), q.pop(), q.empty())

1
2 3 4 True


In [ ]:
class NestedIterator:
    def __init__(self, nested):
        self.stack = [iter(nested)]   # one iterator per list we are inside
        self.nxt = None               # the next integer, ready to hand out
        self._advance()

    def _advance(self):
        self.nxt = None
        while self.stack:
            try:
                item = next(self.stack[-1])
            except StopIteration:
                self.stack.pop()      # this list is finished
                continue
            if isinstance(item, list):
                self.stack.append(iter(item))   # go one level deeper
            else:
                self.nxt = item
                return

    def has_next(self):
        return self.nxt is not None

    def next(self):
        value = self.nxt
        self._advance()
        return value

def flatten(nested):                 # generator version
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

data = [1, [4, [6]], [], 2]
it = NestedIterator(data)
got = []
while it.has_next():
    got.append(it.next())
print(got)
assert got == list(flatten(data))
print(NestedIterator([[], [[]]]).has_next())

[1, 4, 6, 2]
False


## 11.4 Time-Based Designs: Rate Limiter, Time Store and Snapshot Array

In [ ]:
from collections import deque

class RateLimiter:
    def __init__(self, limit, window):
        self.limit, self.window = limit, window
        self.times = deque()                 # allowed timestamps, oldest first

    def allow(self, t):
        while self.times and self.times[0] <= t - self.window:
            self.times.popleft()             # expired: outside (t - window, t]
        if len(self.times) < self.limit:
            self.times.append(t)
            return True
        return False                         # denied requests are not recorded

r = RateLimiter(3, 10)
results = [r.allow(t) for t in (1, 3, 4, 8, 12, 13)]
print(results)
print(list(r.times))
assert results == [True, True, True, False, True, True]

[True, True, True, False, True, True]
[4, 12, 13]


In [ ]:
from bisect import bisect_right
from collections import defaultdict

class TimeMap:
    def __init__(self):
        self.times = defaultdict(list)    # key -> timestamps, increasing
        self.vals = defaultdict(list)     # key -> values, same order

    def set(self, key, value, t):
        self.times[key].append(t)
        self.vals[key].append(value)

    def get(self, key, t):
        ts = self.times.get(key)          # .get: do not create empty entries
        if not ts:
            return ""
        i = bisect_right(ts, t)           # how many timestamps are <= t
        return self.vals[key][i - 1] if i else ""

m = TimeMap()
for t, v in ((1, "a"), (4, "b"), (9, "c"), (12, "d")):
    m.set("k", v, t)
print([m.get("k", t) for t in (0, 1, 3, 6, 9, 100)])
print(repr(m.get("nope", 5)))

['', 'a', 'a', 'b', 'c', 'd']
''


In [ ]:
from bisect import bisect_right

class SnapshotArray:
    def __init__(self, length):
        # history[i] = [(snap_id, value), ...] with increasing snap_id
        self.history = [[(0, 0)] for _ in range(length)]
        self.snap_id = 0

    def set(self, index, val):
        h = self.history[index]
        if h[-1][0] == self.snap_id:
            h[-1] = (self.snap_id, val)      # same snapshot: overwrite
        else:
            h.append((self.snap_id, val))

    def snap(self):
        self.snap_id += 1
        return self.snap_id - 1

    def get(self, index, snap_id):
        h = self.history[index]
        i = bisect_right(h, (snap_id, float("inf"))) - 1
        return h[i][1]

a = SnapshotArray(3)
a.set(0, 5)
s0 = a.snap()
a.set(0, 6)
r1 = a.get(0, s0)
s1 = a.snap()
a.set(1, 9)
s2 = a.snap()
print(s0, s1, s2, r1, a.get(1, s0), a.get(1, s1), a.get(1, s2), a.get(0, s1))
assert (r1, a.get(1, s2), a.get(0, s1)) == (5, 9, 6)

0 1 2 5 0 0 9 6


## 11.5 Randomised Set: Insert, Delete and Random Pick in O(1)

In [ ]:
import random

class RandomSet:
    def __init__(self, seed=0):
        self.items = []          # values, packed with no gaps
        self.pos = {}            # value -> its index in items
        self.rng = random.Random(seed)

    def insert(self, x):
        if x in self.pos:
            return False
        self.pos[x] = len(self.items)
        self.items.append(x)
        return True

    def remove(self, x):
        if x not in self.pos:
            return False
        i = self.pos[x]
        last = self.items[-1]
        self.items[i] = last     # move the last value into the hole
        self.pos[last] = i       # ...and tell the dictionary
        self.items.pop()
        del self.pos[x]          # delete AFTER the fix (x may equal last)
        return True

    def get_random(self):
        return self.items[self.rng.randrange(len(self.items))]

s = RandomSet()
print([s.insert(v) for v in (10, 20, 30, 40, 20)])
print(s.remove(20), s.items, s.pos)
print(s.remove(30), s.items, s.pos)
print(s.remove(99))
draws = {s.get_random() for _ in range(200)}
print(sorted(draws))

[True, True, True, True, False]
True [10, 40, 30] {10: 0, 30: 2, 40: 1}
True [10, 40] {10: 0, 40: 1}
False
[10, 40]


## 11.6 Autocomplete: Trie plus Heap

In [ ]:
import heapq

def build(words):
    root = {}
    for w, count in words.items():
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = count                 # "$" marks a word end and stores its count
    return root

def suggest(root, prefix, k):
    node = root
    for ch in prefix:                     # step 1: walk the prefix
        if ch not in node:
            return []
        node = node[ch]
    found = []                            # step 2: collect all words below
    stack = [(node, prefix)]
    while stack:
        cur, word = stack.pop()
        for ch, child in cur.items():
            if ch == "$":
                found.append((word, child))
            else:
                stack.append((child, word + ch))
    best = heapq.nsmallest(k, found, key=lambda p: (-p[1], p[0]))   # step 3
    return [w for w, _ in best]

root = build({"car": 5, "card": 3, "care": 7, "cat": 2, "dog": 4})
print(suggest(root, "ca", 2))
print(suggest(root, "ca", 10))
print(suggest(root, "", 3))
print(suggest(root, "x", 3))

['care', 'car']
['care', 'car', 'card', 'cat']
['care', 'car', 'dog']
[]


In [ ]:
import heapq

class AutocompleteSystem:
    def __init__(self, sentences, times):
        self.root = {}
        for s, t in zip(sentences, times):
            self._add(s, t)
        self._reset()

    def _add(self, s, t):
        node = self.root
        for ch in s:
            node = node.setdefault(ch, {})
        node["$"] = node.get("$", 0) + t

    def _reset(self):
        self.node, self.typed, self.dead = self.root, "", False

    def input(self, c):
        if c == "#":
            self._add(self.typed, 1)
            self._reset()
            return []
        self.typed += c
        if self.dead or c not in self.node:
            self.dead = True              # nobody starts with this text
            return []
        self.node = self.node[c]          # move ONE step, not from the root
        found, stack = [], [(self.node, self.typed)]
        while stack:
            cur, word = stack.pop()
            for ch, child in cur.items():
                if ch == "$":
                    found.append((word, child))
                else:
                    stack.append((child, word + ch))
        best = heapq.nsmallest(3, found, key=lambda p: (-p[1], p[0]))
        return [w for w, _ in best]

ac = AutocompleteSystem(["i love you", "island", "iroman", "i love leetcode"], [5, 3, 2, 2])
print(ac.input("i"))
print(ac.input(" "))
print(ac.input("a"))
print(ac.input("#"))
print(ac.input("i"))
print(ac.input(" "))

['i love you', 'island', 'i love leetcode']
['i love you', 'i love leetcode']
[]
[]
['i love you', 'island', 'i love leetcode']
['i love you', 'i love leetcode', 'i a']


## 11.7 Union-Find Applications

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))   # everyone starts as their own leader
        self.size = [1] * n            # group size, valid at leaders
        self.groups = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False               # already in the same group
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra            # attach the smaller group to the bigger
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.groups -= 1
        return True

d = DSU(6)
for a, b in ((0, 1), (2, 3), (1, 3), (4, 5), (0, 3)):
    merged = d.union(a, b)
    print((a, b), merged, d.parent, d.groups)

(0, 1) True [0, 0, 2, 3, 4, 5] 5
(2, 3) True [0, 0, 2, 2, 4, 5] 4
(1, 3) True [0, 0, 0, 2, 4, 5] 3
(4, 5) True [0, 0, 0, 2, 4, 4] 2
(0, 3) False [0, 0, 0, 0, 4, 4] 2


In [ ]:
def redundant_edge(edges):
    parent = list(range(len(edges) + 1))   # labels run 1..n, so n + 1 slots

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return [a, b]                  # a and b were already connected
        parent[rb] = ra
    return []

print(redundant_edge([[1, 2], [1, 3], [2, 3]]))
print(redundant_edge([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))

[2, 3]
[1, 4]


In [ ]:
from collections import defaultdict

def merge_accounts(accounts):
    parent = {}                           # email -> parent email
    owner = {}                            # email -> name

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for name, *emails in accounts:
        for e in emails:
            owner[e] = name
            parent[find(e)] = find(emails[0])   # link to the first email

    groups = defaultdict(list)
    for e in owner:
        groups[find(e)].append(e)
    return sorted([owner[r]] + sorted(es) for r, es in groups.items())

accts = [["John", "a@x", "b@x"], ["John", "b@x", "c@x"], ["Mary", "d@x"], ["John", "e@x"]]
result = merge_accounts(accts)
print(result)
assert result == [["John", "a@x", "b@x", "c@x"], ["John", "e@x"], ["Mary", "d@x"]]

[['John', 'a@x', 'b@x', 'c@x'], ['John', 'e@x'], ['Mary', 'd@x']]


## 11.8 Bloom Filters, Consistent Hashing and Thread-Safety Awareness

Some design questions ask whether you *know a structure exists* and when to choose it. This section gives each idea, a small working version, and the sentence to say in an interview.

In [ ]:
import hashlib
import math

class BloomFilter:
    def __init__(self, m, k):
        self.m, self.k = m, k             # m bits, k hash functions
        self.bits = bytearray(m)          # one byte per bit, for clarity

    def _positions(self, item):
        for i in range(self.k):           # k different hashes: salt with i
            h = hashlib.md5(f"{i}:{item}".encode()).hexdigest()
            yield int(h, 16) % self.m

    def add(self, item):
        for p in self._positions(item):
            self.bits[p] = 1

    def might_contain(self, item):
        return all(self.bits[p] for p in self._positions(item))

bf = BloomFilter(m=1000, k=3)
for i in range(100):
    bf.add(f"user{i}")
assert all(bf.might_contain(f"user{i}") for i in range(100))   # never a false "no"
false_yes = sum(bf.might_contain(f"guest{i}") for i in range(1000))
theory = (1 - math.exp(-3 * 100 / 1000)) ** 3                  # (1 - e^(-kn/m))^k
print(false_yes, round(theory * 1000, 1))

15 17.4


In [ ]:
import hashlib
from bisect import bisect_right

def h(s):
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % (2 ** 32)

class Ring:
    def __init__(self, nodes, vnodes=100):
        self.vnodes = vnodes
        self.points = []                  # sorted positions on the ring
        self.owner = {}                   # position -> server name
        for n in nodes:
            self.add(n)

    def add(self, node):
        for v in range(self.vnodes):      # many virtual spots per server
            p = h(f"{node}#{v}")
            self.owner[p] = node
            self.points.append(p)
        self.points.sort()

    def get(self, key):
        i = bisect_right(self.points, h(key)) % len(self.points)   # wrap around
        return self.owner[self.points[i]]

keys = [f"key{i}" for i in range(2000)]
ring = Ring(["A", "B", "C"])
before = {k: ring.get(k) for k in keys}
ring.add("D")
after = {k: ring.get(k) for k in keys}
moved = [k for k in keys if before[k] != after[k]]
print("ring moved:", len(moved), "of", len(keys))
print("all moved keys went to D:", all(after[k] == "D" for k in moved))
mod3 = {k: h(k) % 3 for k in keys}
mod4 = {k: h(k) % 4 for k in keys}
print("mod N moved:", sum(mod3[k] != mod4[k] for k in keys), "of", len(keys))

ring moved: 557 of 2000
all moved keys went to D: True
mod N moved: 1525 of 2000


In [ ]:
import threading
from collections import OrderedDict

class SafeLRU:
    def __init__(self, capacity):
        self.cap = capacity
        self.data = OrderedDict()
        self.lock = threading.Lock()       # one lock guards the whole structure

    def get(self, key):
        with self.lock:                    # the whole method runs as one unit
            if key not in self.data:
                return -1
            self.data.move_to_end(key)
            return self.data[key]

    def put(self, key, value):
        with self.lock:
            if key in self.data:
                self.data.move_to_end(key)
            self.data[key] = value
            if len(self.data) > self.cap:
                self.data.popitem(last=False)

cache = SafeLRU(8)

def worker(base):
    for i in range(500):
        cache.put(base + i % 10, i)
        cache.get(base + (i + 3) % 10)

threads = [threading.Thread(target=worker, args=(b,)) for b in (0, 100, 200, 300)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(len(cache.data))
assert len(cache.data) == 8

8


## Chapter recap

## Practice set

Try each problem on paper first. They run from easy to hard and mix concepts, hand traces, predict-the-output, find-the-bug, code and interview questions. Code for predict and find-the-bug problems is shown without output so you can run it yourself.

In [ ]:
from collections import OrderedDict

d = OrderedDict()
d["a"] = 1
d["b"] = 2
d["c"] = 3
d.move_to_end("a")
d.popitem(last=False)
print(list(d))

In [ ]:
from collections import OrderedDict

class BadLRU:
    def __init__(self, cap):
        self.cap, self.d = cap, OrderedDict()

    def get(self, k):
        return self.d.get(k, -1)

    def put(self, k, v):
        if k in self.d:
            self.d.move_to_end(k)
        self.d[k] = v
        if len(self.d) > self.cap:
            self.d.popitem(last=False)

c = BadLRU(2)
c.put(1, 1)
c.put(2, 2)
print(c.get(1))
c.put(3, 3)
print(c.get(1))

In [ ]:
class BadMinStack:
    def __init__(self):
        self.items, self.mins = [], []

    def push(self, x):
        self.items.append(x)
        if not self.mins or x < self.mins[-1]:
            self.mins.append(x)

    def pop(self):
        x = self.items.pop()
        if x == self.mins[-1]:
            self.mins.pop()
        return x

    def get_min(self):
        return self.mins[-1]

s = BadMinStack()
s.push(2)
s.push(2)
s.pop()
try:
    print(s.get_min())
except IndexError:
    print("crash: mins is empty")

In [ ]:
inbox, outbox = [], []

def push(x):
    inbox.append(x)

def pop():
    if not outbox:
        while inbox:
            outbox.append(inbox.pop())
    return outbox.pop()

push(1)
push(2)
a = pop()
push(3)
b = pop()
c = pop()
print(a, b, c, inbox, outbox)

In [ ]:
from bisect import bisect_left

ts = [1, 4, 9]
vals = ["a", "b", "c"]

def get(t):
    i = bisect_left(ts, t)
    return vals[i - 1] if i else ""

print(repr(get(9)), repr(get(5)), repr(get(1)))

In [ ]:
from collections import OrderedDict

class TTLCache:
    def __init__(self, capacity, ttl):
        self.cap, self.ttl = capacity, ttl
        self.data = OrderedDict()               # key -> (value, expires_at)

    def get(self, key, now):
        item = self.data.get(key)
        if item is None:
            return -1
        value, expires = item
        if expires <= now:                      # expired: treat as missing
            del self.data[key]
            return -1
        self.data.move_to_end(key)
        return value

    def put(self, key, value, now):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = (value, now + self.ttl)
        if len(self.data) > self.cap:
            self.data.popitem(last=False)

c = TTLCache(2, 10)
c.put("a", 1, now=0)
c.put("b", 2, now=1)
print(c.get("a", 5), c.get("a", 10), c.get("b", 10), c.get("b", 11))

1 -1 2 -1


In [ ]:
class HitCounter300:
    def __init__(self):
        self.stamp = [0] * 300       # the second each bucket currently holds
        self.hits = [0] * 300

    def hit(self, t):
        i = t % 300
        if self.stamp[i] != t:       # bucket holds an old second: reset it
            self.stamp[i] = t
            self.hits[i] = 0
        self.hits[i] += 1

    def count(self, t):
        return sum(h for s, h in zip(self.stamp, self.hits) if t - 300 < s <= t)

h = HitCounter300()
for t in (1, 2, 3):
    h.hit(t)
r = [h.count(4)]
h.hit(300)
r.append(h.count(300))
h.hit(301)
r.append(h.count(301))
r.append(h.count(302))
print(r)
assert r == [3, 4, 4, 3]

[3, 4, 4, 3]


In [ ]:
from collections import deque

class RoundRobin:
    def __init__(self, lists):
        self.q = deque((lst, 0) for lst in lists if lst)

    def has_next(self):
        return bool(self.q)

    def next(self):
        lst, i = self.q.popleft()
        if i + 1 < len(lst):
            self.q.append((lst, i + 1))   # go to the back of the line
        return lst[i]

it = RoundRobin([[1, 2, 3], [4], [], [5, 6]])
out = []
while it.has_next():
    out.append(it.next())
print(out)
assert out == [1, 4, 5, 2, 6, 3]

[1, 4, 5, 2, 6, 3]


In [ ]:
same = [r for r in range(20) if r % 4 == r % 5]
print(same, len(same), "of 20")
print(sum(k % 4 == k % 5 for k in range(100000)), "of 100000")

[0, 1, 2, 3] 4 of 20
20000 of 100000


In [ ]:
def add_word(counts, w):
    if w in counts:
        counts[w] += 1
    else:
        counts[w] = 1

In [ ]:
import threading

counts = {}
lock = threading.Lock()

def add_word(w):
    with lock:                          # check and update as one unit
        counts[w] = counts.get(w, 0) + 1

def work():
    for _ in range(1000):
        add_word("x")

threads = [threading.Thread(target=work) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(counts)

{'x': 4000}


In [ ]:
import heapq

class Leaderboard:
    def __init__(self):
        self.score = {}

    def add_score(self, player, s):
        self.score[player] = self.score.get(player, 0) + s

    def top(self, k):
        return sum(heapq.nlargest(k, self.score.values()))

    def reset(self, player):
        self.score.pop(player, None)

lb = Leaderboard()
for p, s in ((1, 73), (2, 56), (3, 39), (4, 51), (5, 4)):
    lb.add_score(p, s)
a = lb.top(1)
lb.reset(1)
lb.reset(2)
lb.add_score(2, 51)
print(a, lb.top(3))
assert (a, lb.top(3)) == (73, 141)

73 141


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch11.html